# Texture a mesh on Colab (Hunyuan3D-2 Paint)

Wraps a **high-quality, multi-view-consistent UV texture** around an EXISTING mesh (e.g. your
TripoSG `.glb`) from a SINGLE front reference image, using
[Hunyuan3D-2 Paint](https://github.com/Tencent-Hunyuan/Hunyuan3D-2) on a Colab GPU (T4 / L4 / A100).

Unlike a flat front-projection, Paint runs **synchronized multi-view diffusion**: it renders the
mesh from 6 viewpoints, diffuses them together (so they agree -> no seams/Janus), injects your
reference into every view, then back-projects them into one baked UV albedo -- the front stays
faithful, the sides/back are synthesized coherently.

### How to run
1. **Runtime -> Change runtime type -> GPU** (T4, or L4/A100 on Colab Pro), then **Save**.
2. **Runtime -> Run all.** At the upload cell, pick **BOTH** your mesh `.glb` and your reference image.

### Notes
- The CUDA op builds for all Colab GPU archs (T4/L4/A100), and the paint cell **auto-picks the texture
  resolution from VRAM**: 2048 on an L4/A100 (Colab Pro), 1024 on a 16 GB T4. (Drop to 768 if a T4 OOMs.)
- The 2.0 Paint model can leave a **faint seam** (the seam-fix RoPE is a 2.1 feature) -- a small Blender touch-up.
- Geometry is untouched; this only adds the texture. First run ~6-8 min (build + ~2 GB weights).

## 1. Check the GPU (T4 / L4 / A100 — any works)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. Install Hunyuan3D-2 Paint (isolated venv)

Builds an isolated venv, compiles the two texgen ops (`custom_rasterizer` CUDA + `mesh_processor`
pybind11), pins a torch-2.4-compatible HF stack. Ends by printing `ops OK` + `paint env OK | cuda True`.

In [ ]:
import os
!pip install -q virtualenv
!virtualenv -q /content/hyenv --python=python3
PY = "/content/hyenv/bin/python"; PIP = "/content/hyenv/bin/pip"
# torch FIRST (cu121) so the CUDA op links against it.
!$PIP install -q torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121
![ -d /content/Hunyuan3D-2 ] || git clone https://github.com/Tencent-Hunyuan/Hunyuan3D-2 /content/Hunyuan3D-2
# Pin a mid-2024 / torch-2.4 HF stack. huggingface_hub<0.26 is load-bearing (diffusers<=0.30 imports
# cached_download, removed in 0.26). Do NOT use the repo's unpinned requirements.txt (2025 wheels break torch 2.4).
!$PIP install -q "diffusers==0.30.2" "transformers==4.46.3" "huggingface_hub==0.25.2" "accelerate==0.34.2" "peft==0.13.2" "tokenizers<0.21" "safetensors>=0.4.3" "numpy<2" "opencv-python" "einops" "omegaconf" "tqdm" "trimesh" "pymeshlab" "pygltflib" "xatlas" "rembg==2.0.59" "onnxruntime==1.19.2" "Pillow" "scipy" "ninja" "pybind11"
# register hy3dgen without clobbering the pins
!cd /content/Hunyuan3D-2 && $PIP install -q -e . --no-deps
# op A: custom_rasterizer (CUDA -> site-packages; absolute import). Build for ALL Colab GPU archs
# + PTX so it runs whatever Pro assigns: T4=7.5, V100=7.0, A100=8.0, L4=8.9.
!cd /content/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer && CUDA_HOME=/usr/local/cuda TORCH_CUDA_ARCH_LIST="7.0;7.5;8.0;8.9+PTX" /content/hyenv/bin/python setup.py install 2>&1 | tail -3
# op B: differentiable_renderer/mesh_processor (pybind11, NO nvcc). MUST be in-place (relative import).
!cd /content/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer && /content/hyenv/bin/python setup.py build_ext --inplace 2>&1 | tail -3
# LOUD verify both ops + the paint pipeline import.
!cd /content/Hunyuan3D-2 && /content/hyenv/bin/python -c "import sys; sys.path.insert(0,'.'); import torch, custom_rasterizer; from hy3dgen.texgen.differentiable_renderer.mesh_processor import meshVerticeInpaint; from hy3dgen.texgen import Hunyuan3DPaintPipeline; print('ops OK'); print('paint env OK | cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0))"

## 3. Upload your mesh + reference image

Pick **BOTH** at once: your existing mesh (`.glb`/`.obj`) and the front reference image (`.png`/`.webp`/`.jpg`).

In [ ]:
from google.colab import files
import shutil, os
up = files.upload()   # select BOTH the mesh and the reference image
MESH_PATH = None; IMG_PATH = None
for name in list(up):
    ext = os.path.splitext(name)[1].lower()
    if ext in ('.glb', '.obj', '.ply'):
        MESH_PATH = '/content/triposg_out' + ext; shutil.move(name, MESH_PATH)
    else:
        IMG_PATH = '/content/ref_front' + ext; shutil.move(name, IMG_PATH)
print('mesh :', MESH_PATH)
print('image:', IMG_PATH)
assert MESH_PATH and IMG_PATH, 'Upload BOTH a mesh (.glb/.obj) AND a reference image.'


## 4. Paint the texture

Runs Hunyuan3D-Paint **inside the venv** (subprocess). `RES=1024` is T4-safe; drop to `768` if it OOMs.
First run downloads ~2 GB of Paint weights, so this cell sits for a few minutes.

In [ ]:
import os, glob, subprocess
PY = "/content/hyenv/bin/python"

# Auto-pick texture resolution from VRAM: 2048 on L4/A100 (Pro), 1024 on a 16GB T4.
try:
    vram = int(subprocess.run(["nvidia-smi","--query-gpu=memory.total","--format=csv,noheader,nounits"],
                              capture_output=True, text=True).stdout.split("\n")[0])
except Exception:
    vram = 16000
RES = 2048 if vram > 20000 else 1024   # override manually if you like (768 if a T4 OOMs)

MESH = glob.glob("/content/triposg_out.*"); MESH = MESH[0] if MESH else None
IMG  = glob.glob("/content/ref_front.*");  IMG  = IMG[0] if IMG else None
assert MESH and IMG, "Run the Upload cell (3) first."
print(f"vram={vram}MB -> RES={RES} | mesh: {MESH} | image: {IMG}")
print("First run downloads ~2GB of weights -> a few min with no output, then it prints all at once.\n")

runner = '\nimport os, sys, traceback\nsys.path.insert(0, "/content/Hunyuan3D-2")\ntry:\n    import torch, trimesh\n    from PIL import Image\n    from hy3dgen.texgen import Hunyuan3DPaintPipeline\n    from hy3dgen.rembg import BackgroundRemover\n    mesh_path, img_path, out_path, res = sys.argv[1], sys.argv[2], sys.argv[3], int(sys.argv[4])\n    loaded = trimesh.load(mesh_path, force="mesh")\n    mesh = loaded if isinstance(loaded, trimesh.Trimesh) else loaded.dump(concatenate=True)\n    image = Image.open(img_path).convert("RGB")\n    image = BackgroundRemover()(image)\n    if image.mode != "RGBA":\n        image = image.convert("RGBA")\n    pipeline = Hunyuan3DPaintPipeline.from_pretrained("tencent/Hunyuan3D-2", subfolder="hunyuan3d-paint-v2-0-turbo")\n    pipeline.config.render_size = res\n    pipeline.config.texture_size = res\n    for k in ("delight_model", "multiview_model"):\n        try:\n            pipeline.models[k].pipeline.enable_vae_slicing()\n            pipeline.models[k].pipeline.enable_vae_tiling()\n        except Exception as e:\n            print("vae slice/tile skipped for", k, ":", e)\n    torch.cuda.empty_cache()\n    print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "| painting at", res)\n    mesh = pipeline(mesh, image=image)\n    mesh.export(out_path)\n    print("DONE ->", out_path)\nexcept Exception:\n    traceback.print_exc(); sys.exit(1)\n'
with open("/content/run_paint.py", "w") as f:
    f.write(runner)

# Capture + print so the output/traceback shows in the CELL (Colab hides raw subprocess fds).
ret = subprocess.run([PY, "/content/run_paint.py", MESH, IMG, "/content/textured.glb", str(RES)],
                     capture_output=True, text=True)
print(ret.stdout)
if ret.stderr:
    print("----- venv stderr -----\n" + ret.stderr)
GLB = "/content/textured.glb"
if ret.returncode != 0 or not os.path.exists(GLB):
    tail = "\n".join((ret.stdout + "\n" + ret.stderr).strip().splitlines()[-60:])
    raise RuntimeError("Hunyuan-Paint failed -- real error:\n" + tail)
print("textured:", GLB)

## 5. Download the textured mesh

In [ ]:
import os
from google.colab import files
assert os.path.exists("/content/textured.glb"), "Run the Paint cell (4) first."
print("size:", round(os.path.getsize("/content/textured.glb")/1e6, 1), "MB")
files.download("/content/textured.glb")


## 6. Back in the repo

Drop the textured `.glb` into `runs/`, then in Blender: decimate to the Roblox tri budget, center/scale,
add attachments, validate, export FBX:

```bash
roblox-ugc autoprep runs/shark/textured.glb --out runs/shark/prepped.fbx --category Hat
roblox-ugc inspect runs/shark/prepped.fbx --out runs/shark/report.json
roblox-ugc validate runs/shark/report.json --target accessory --category Hat
```

- Texture is a baked **UV albedo** (front faithful; sides/back synthesized). A faint seam, if any, is a quick Blender touch-up.
- **License:** Tencent Hunyuan Community License -- commercial use permitted under 1M MAU (excludes EU/UK/South Korea). Review before large-scale resale.

> Note: a **TPU** can't run this -- custom CUDA kernels. Use the **T4 GPU** runtime.